## Iniciando o Spark e importando bibliotecas

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import to_date, lit, date_format

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("BookBureau") \
    .master("spark://spark-master:7077") \
    .getOrCreate()




Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/17 18:58:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
bureau = spark.read \
    .parquet("/data/processed/bureau")

bureau.createOrReplaceTempView("bureau")
bureau.show(5, truncate=False)

+-------------+------------------------+--------------------------+----------------------+----------------------+------------------------------+-----------------------------+-----------------------------+----------------------------+---------------------------------+-----------------------------+-------------------------------+------------------------------+-------------------------+---------------------------------+----------------------+-------------------+-----------------+----------+
|PK_AGG_BUREAU|FBC_CREDIT_ACTIVE_BUREAU|FBC_CREDIT_CURRENCY_BUREAU|FBC_CREDIT_TYPE_BUREAU|FVL_DAYS_CREDIT_BUREAU|FVL_DAYS_CREDIT_ENDDATE_BUREAU|FVL_DAYS_CREDIT_UPDATE_BUREAU|FVL_CREDIT_DAY_OVERDUE_BUREAU|FVL_DAYS_ENDDATE_FACT_BUREAU|FVL_AMT_CREDIT_MAX_OVERDUE_BUREAU|FVL_CNT_CREDIT_PROLONG_BUREAU|FVL_AMT_CREDIT_SUM_LIMIT_BUREAU|FVL_AMT_CREDIT_SUM_DEBT_BUREAU|FVL_AMT_CREDIT_SUM_BUREAU|FVL_AMT_CREDIT_SUM_OVERDUE_BUREAU|FVL_AMT_ANNUITY_BUREAU|PK_DATREF          |FBC_STATUS_BUREAU|PK_DATPART|
+-------------

In [3]:
bureau.count()

24179741

### Vamos explorar os domínios da tabela transacional
- Olhar as volumetrias das categorias das variáves que são regras de negócio
- FBC_CREDIT_ACTIVE
- FBC_CREDIT_CURRENCY
- FBC_CREDIT_TYPE

Após essa exploração seremos capazes de construi variáveis explicativas pra Machine Learning

In [4]:
qtd_linhas = bureau.count()

In [5]:
spark.sql("""
            Select
                FBC_CREDIT_TYPE_BUREAU,
                count(*) as VOLUME,
                round(100*(count(*)/{}),2) as VOL_PERCENT
            from 
                bureau
            group by 
                FBC_CREDIT_TYPE_BUREAU
            order by 
                VOLUME desc
""".format(qtd_linhas)).show(50,False)

+--------------------------------------------+--------+-----------+
|FBC_CREDIT_TYPE_BUREAU                      |VOLUME  |VOL_PERCENT|
+--------------------------------------------+--------+-----------+
|Consumer credit                             |18333667|75.82      |
|Credit card                                 |5019228 |20.76      |
|Car loan                                    |446901  |1.85       |
|Mortgage                                    |251213  |1.04       |
|Microloan                                   |48856   |0.2        |
|Loan for business development               |42537   |0.18       |
|Another type of loan                        |17271   |0.07       |
|Unknown type of loan                        |13129   |0.05       |
|Loan for working capital replenishment      |4902    |0.02       |
|Cash loan (non-earmarked)                   |1445    |0.01       |
|Loan for the purchase of equipment          |291     |0.0        |
|Real estate loan                            |24

In [6]:
spark.sql("""
            Select
                FBC_CREDIT_ACTIVE_BUREAU,
                count(*) as VOLUME,
                round(100*(count(*)/{}),2) as VOL_PERCENT
            from 
                bureau
            group by 
                FBC_CREDIT_ACTIVE_BUREAU
            order by 
                VOLUME desc
""".format(qtd_linhas)).show(50,False)

+------------------------+--------+-----------+
|FBC_CREDIT_ACTIVE_BUREAU|VOLUME  |VOL_PERCENT|
+------------------------+--------+-----------+
|Closed                  |18511853|76.56      |
|Active                  |5545692 |22.94      |
|Sold                    |122096  |0.5        |
|Bad debt                |100     |0.0        |
+------------------------+--------+-----------+



In [7]:
spark.sql("""
            select 
                FBC_CREDIT_CURRENCY_BUREAU,
                count(*) as VOLUME,
                round(100*(count(*)/{}),1) as VOL_PERCENT
            from 
                bureau
            group by 
                FBC_CREDIT_CURRENCY_BUREAU
            order by 
                VOLUME desc
""".format(qtd_linhas)).show(50,False)

[Stage 15:=====================================================>  (24 + 1) / 25]

+--------------------------+--------+-----------+
|FBC_CREDIT_CURRENCY_BUREAU|VOLUME  |VOL_PERCENT|
+--------------------------+--------+-----------+
|currency 1                |24148414|99.9       |
|currency 2                |27938   |0.1        |
|currency 3                |3212    |0.0        |
|currency 4                |177     |0.0        |
+--------------------------+--------+-----------+



In [8]:
bureau_etl_01 = spark.sql("""
    SELECT
        PK_AGG_BUREAU,

        --Quantidade de créditos por status
        sum(case when FBC_CREDIT_ACTIVE_BUREAU = 'Active' then 1 else 0 end) as QT_CRED_Active_BUREAU,       -- Créditos em aberto
        sum(case when FBC_CREDIT_ACTIVE_BUREAU = 'Sold' then 1 else 0 end) as QT_CRED_Sold_BUREAU,           -- Créditos vendidos a terceiros
        sum(case when FBC_CREDIT_ACTIVE_BUREAU = 'Closed' then 1 else 0 end) as QT_CRED_Closed_BUREAU,       -- Créditos encerrados
        sum(case when FBC_CREDIT_ACTIVE_BUREAU = 'Bad debt' then 1 else 0 end) as QT_CRED_Bad_debt_BUREAU,   -- Créditos inadimplentes

        --Quantidade de créditos por tipo
        sum(case when FBC_CREDIT_TYPE_BUREAU = 'Microloan' then 1 else 0 end) as QT_CRED_Microloan_BUREAU,
        sum(case when FBC_CREDIT_TYPE_BUREAU = 'Credit card' then 1 else 0 end) as QT_CRED_CreditCard_BUREAU,
        sum(case when FBC_CREDIT_TYPE_BUREAU = 'Consumer credit' then 1 else 0 end) as QT_CRED_ConsumerCredit_BUREAU,
        sum(case when FBC_CREDIT_TYPE_BUREAU = 'Car loan' then 1 else 0 end) as QT_CRED_CarLoan_BUREAU,
        sum(case when FBC_CREDIT_TYPE_BUREAU = 'Mortgage' then 1 else 0 end) as QT_CRED_Mortgage_BUREAU,

        --Quantidade de créditos por moeda
        sum(case when FBC_CREDIT_CURRENCY_BUREAU = 'currency 2' then 1 else 0 end) as QT_CRED_currency2_BUREAU,
        sum(case when FBC_CREDIT_CURRENCY_BUREAU = 'currency 1' then 1 else 0 end) as QT_CRED_currency1_BUREAU,

        --Valor médio de crédito por status
        avg(case when FBC_CREDIT_ACTIVE_BUREAU = 'Active' then FVL_AMT_CREDIT_SUM_BUREAU else null end) as AVG_AMT_FBC_CREDIT_ACTIVE_BUREAU,
        avg(case when FBC_CREDIT_ACTIVE_BUREAU = 'Sold' then FVL_AMT_CREDIT_SUM_BUREAU else null end) as AVG_AMT_CREDIT_Sold_BUREAU,
        avg(case when FBC_CREDIT_ACTIVE_BUREAU = 'Closed' then FVL_AMT_CREDIT_SUM_BUREAU else null end) as AVG_AMT_CREDIT_Closed_BUREAU,
        avg(case when FBC_CREDIT_ACTIVE_BUREAU = 'Bad debt' then FVL_AMT_CREDIT_SUM_BUREAU else null end) as AVG_AMT_CREDIT_BadDebt_BUREAU,

        --Valor médio de crédito por tipo
        avg(case when FBC_CREDIT_TYPE_BUREAU = 'Microloan' then FVL_AMT_CREDIT_SUM_BUREAU else null end) as AVG_AMT_CREDIT_Microloan_BUREAU,
        avg(case when FBC_CREDIT_TYPE_BUREAU = 'Credit card' then FVL_AMT_CREDIT_SUM_BUREAU else null end) as AVG_AMT_CREDIT_Creditcard_BUREAU,
        avg(case when FBC_CREDIT_TYPE_BUREAU = 'Consumer credit' then FVL_AMT_CREDIT_SUM_BUREAU else null end) as AVG_AMT_CREDIT_Consumercredit_BUREAU,
        avg(case when FBC_CREDIT_TYPE_BUREAU = 'Car loan' then FVL_AMT_CREDIT_SUM_BUREAU else null end) as AVG_AMT_CREDIT_Carloan_BUREAU,
        avg(case when FBC_CREDIT_TYPE_BUREAU = 'Mortgage' then FVL_AMT_CREDIT_SUM_BUREAU else null end) as AVG_AMT_CREDIT_Mortgage_BUREAU,

        --Valor médio de crédito por moeda
        avg(case when FBC_CREDIT_CURRENCY_BUREAU = 'currency 1' then FVL_AMT_CREDIT_SUM_BUREAU else null end) as AVG_AMT_FBC_CREDIT_CURRENCY1_BUREAU,

        --Estatísticas de valores totais de crédito
        min(FVL_AMT_CREDIT_SUM_BUREAU) as MIN_FVL_AMT_CREDIT_SUM_BUREAU,
        max(FVL_AMT_CREDIT_SUM_BUREAU) as MAX_FVL_AMT_CREDIT_SUM_BUREAU,
        avg(FVL_AMT_CREDIT_SUM_BUREAU) as AVG_FVL_AMT_CREDIT_SUM_BUREAU,

        --Informações de datas desde a concessão
        min(FVL_DAYS_CREDIT_BUREAU) as MIN_FVL_DAYS_CREDIT_BUREAU,
        max(FVL_DAYS_CREDIT_BUREAU) as MAX_FVL_DAYS_CREDIT_BUREAU,
        avg(FVL_DAYS_CREDIT_BUREAU) as AVG_FVL_DAYS_CREDIT_BUREAU,

        --Informações sobre valores em dívida
        min(FVL_AMT_CREDIT_SUM_DEBT_BUREAU) as MIN_FVL_AMT_CREDIT_SUM_DEBT_BUREAU,
        max(FVL_AMT_CREDIT_SUM_DEBT_BUREAU) as MAX_FVL_AMT_CREDIT_SUM_DEBT_BUREAU,
        avg(FVL_AMT_CREDIT_SUM_DEBT_BUREAU) as AVG_FVL_AMT_CREDIT_SUM_DEBT_BUREAU,

        --Valores em atraso
        min(FVL_AMT_CREDIT_SUM_OVERDUE_BUREAU) as MIN_FVL_AMT_CREDIT_SUM_OVERDUE_BUREAU,
        max(FVL_AMT_CREDIT_SUM_OVERDUE_BUREAU) as MAX_FVL_AMT_CREDIT_SUM_OVERDUE_BUREAU,
        avg(FVL_AMT_CREDIT_SUM_OVERDUE_BUREAU) as AVG_FVL_AMT_CREDIT_SUM_OVERDUE_BUREAU,

        --Datas da última atualização do crédito
        min(FVL_DAYS_CREDIT_UPDATE_BUREAU) as MIN_FVL_DAYS_CREDIT_UPDATE_BUREAU,
        max(FVL_DAYS_CREDIT_UPDATE_BUREAU) as MAX_FVL_DAYS_CREDIT_UPDATE_BUREAU,
        avg(FVL_DAYS_CREDIT_UPDATE_BUREAU) as AVG_FVL_DAYS_CREDIT_UPDATE_BUREAU,

        --Quantidade de prorrogações
        min(FVL_CNT_CREDIT_PROLONG_BUREAU) as MIN_FVL_CNT_CREDIT_PROLONG_BUREAU,
        max(FVL_CNT_CREDIT_PROLONG_BUREAU) as MAX_FVL_CNT_CREDIT_PROLONG_BUREAU,
        avg(FVL_CNT_CREDIT_PROLONG_BUREAU) as AVG_FVL_CNT_CREDIT_PROLONG_BUREAU,

        --Estatísticas de anuidade
        min(FVL_AMT_ANNUITY_BUREAU) as MIN_FVL_AMT_ANNUITY_BUREAU,
        max(FVL_AMT_ANNUITY_BUREAU) as MAX_FVL_AMT_ANNUITY_BUREAU,
        avg(FVL_AMT_ANNUITY_BUREAU) as AVG_FVL_AMT_ANNUITY_BUREAU,

        --Valor médio de créditos ativos nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FBC_CREDIT_ACTIVE_BUREAU = 'Active' and PK_DATPART in (202310, 202311, 202312)
                 then FVL_AMT_CREDIT_SUM_BUREAU else null end) as AVG_AMT_CREDIT_ACTIVE_U3M_BUREAU,

        --Valor médio de créditos ativos nos últimos 6 meses (jul a dez 2023)
        avg(case when FBC_CREDIT_ACTIVE_BUREAU = 'Active' and PK_DATPART in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_CREDIT_SUM_BUREAU else null end) as AVG_AMT_CREDIT_ACTIVE_U6M_BUREAU,

        --Valor médio de créditos ativos nos últimos 9 meses (abr a dez 2023)
        avg(case when FBC_CREDIT_ACTIVE_BUREAU = 'Active' and PK_DATPART in (202304, 202305, 202306, 202307, 202308, 202309,
                                                                      202310, 202311, 202312)
                 then FVL_AMT_CREDIT_SUM_BUREAU else null end) as AVG_AMT_CREDIT_ACTIVE_U9M_BUREAU,

        --Valor médio de créditos ativos nos últimos 12 meses (jan a dez 2023)
        avg(case when FBC_CREDIT_ACTIVE_BUREAU = 'Active' and PK_DATPART in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_CREDIT_SUM_BUREAU else null end) as AVG_AMT_CREDIT_ACTIVE_U12M_BUREAU,


        --NOVAS VARIÁVEIS DERIVADAS:

        -- Índices financeiros
        avg(FVL_AMT_CREDIT_SUM_DEBT_BUREAU / nullif(FVL_AMT_CREDIT_SUM_BUREAU, 0)) as AVG_RATIO_DEBT_TO_TOTAL_BUREAU,
        avg(FVL_AMT_CREDIT_SUM_OVERDUE_BUREAU / nullif(FVL_AMT_CREDIT_SUM_BUREAU, 0)) as AVG_RATIO_OVERDUE_TO_TOTAL_BUREAU,
        avg(FVL_AMT_CREDIT_MAX_OVERDUE_BUREAU / nullif(FVL_AMT_CREDIT_SUM_BUREAU, 0)) as AVG_RATIO_MAXOVERDUE_TO_TOTAL_BUREAU,
        avg(FVL_AMT_CREDIT_SUM_DEBT_BUREAU / nullif(FVL_AMT_CREDIT_SUM_LIMIT_BUREAU, 0)) as AVG_RATIO_DEBT_TO_LIMIT_BUREAU,
        avg(FVL_AMT_ANNUITY_BUREAU / nullif(FVL_AMT_CREDIT_SUM_DEBT_BUREAU, 0)) as AVG_RATIO_ANNUITY_TO_DEBT_BUREAU,

        -- Frequência de prorrogação
        sum(case when FVL_CNT_CREDIT_PROLONG_BUREAU > 0 then 1 else 0 end) as QT_CONTRACTS_PROLONGED_BUREAU,
        avg(case when FVL_CNT_CREDIT_PROLONG_BUREAU > 0 then 1.0 else 0 end) as RATIO_HAS_PROLONG_BUREAU,

        -- Dias médios em atraso
        avg(FVL_CREDIT_DAY_OVERDUE_BUREAU) as AVG_OVERDUE_DAYS_BUREAU,

        -- Tempo de contrato
        avg(FVL_DAYS_CREDIT_ENDDATE_BUREAU - FVL_DAYS_CREDIT_BUREAU) as AVG_DAYS_TO_END_BUREAU,
        avg(FVL_DAYS_ENDDATE_FACT_BUREAU - FVL_DAYS_CREDIT_ENDDATE_BUREAU) as AVG_DIFF_REAL_VS_ESTIMATED_BUREAU,

        -- Pagamentos antecipados
        avg(case when FVL_DAYS_ENDDATE_FACT_BUREAU < FVL_DAYS_CREDIT_ENDDATE_BUREAU then 1 else 0 end) as RATIO_EARLY_REPAID_BUREAU

    FROM bureau
    GROUP BY PK_AGG_BUREAU
""")

bureau_etl_01.createOrReplaceTempView("bureau_etl_01")
bureau_etl_01.count()

25/07/17 18:59:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

134542

In [9]:
bureau_etl_01.show(5)

[Stage 26:>                                                         (0 + 1) / 1]

+-------------+---------------------+-------------------+---------------------+-----------------------+------------------------+-------------------------+-----------------------------+----------------------+-----------------------+------------------------+------------------------+--------------------------------+--------------------------+----------------------------+-----------------------------+-------------------------------+--------------------------------+------------------------------------+-----------------------------+------------------------------+-----------------------------------+-----------------------------+-----------------------------+-----------------------------+--------------------------+--------------------------+--------------------------+----------------------------------+----------------------------------+----------------------------------+-------------------------------------+-------------------------------------+-------------------------------------+-----------

# 📘 Dicionário de Variáveis - Book Bureau

| Variável                        | Descrição                                                                 |
|--------------------------------|---------------------------------------------------------------------------|
| `PK_AGG_BUREAU`                       | Identificador do cliente para agregação dos dados                        |
| `QT_CRED_Active_BUREAU`               | Quantidade de créditos em aberto no momento da consulta                  |
| `QT_CRED_Sold_BUREAU`                 | Quantidade de créditos vendidos a terceiros                              |
| `QT_CRED_Closed_BUREAU`               | Quantidade de créditos já encerrados                                     |
| `QT_CRED_Bad_debt_BUREAU`            | Quantidade de créditos classificados como inadimplentes                  |
| `QT_CRED_Microloan_BUREAU`           | Número de contratos do tipo microcrédito                                 |
| `QT_CRED_CreditCard_BUREAU`          | Número de cartões de crédito                                              |
| `QT_CRED_ConsumerCredit_BUREAU`      | Número de créditos ao consumidor                                          |
| `QT_CRED_CarLoan_BUREAU`             | Número de financiamentos de veículos                                     |
| `QT_CRED_Mortgage_BUREAU`            | Número de financiamentos imobiliários                                    |
| `QT_CRED_currency1_BUREAU`           | Quantidade de créditos em moeda local (currency 1)                       |
| `AVG_AMT_FBC_CREDIT_ACTIVE_BUREAU`   | Valor médio dos créditos ativos                                          |
| `AVG_AMT_CREDIT_Sold_BUREAU`         | Valor médio dos créditos vendidos                                        |
| `AVG_AMT_CREDIT_Closed_BUREAU`       | Valor médio dos créditos encerrados                                      |
| `AVG_AMT_CREDIT_BadDebt_BUREAU`      | Valor médio dos créditos inadimplentes                                   |
| `AVG_AMT_CREDIT_Microloan_BUREAU`    | Valor médio dos contratos de microcrédito                                |
| `AVG_AMT_CREDIT_Creditcard_BUREAU`   | Valor médio dos cartões de crédito                                       |
| `AVG_AMT_CREDIT_Consumercredit_BUREAU`| Valor médio dos créditos ao consumidor                                   |
| `AVG_AMT_CREDIT_Carloan_BUREAU`      | Valor médio dos financiamentos de veículos                               |
| `AVG_AMT_CREDIT_Mortgage_BUREAU`     | Valor médio dos financiamentos imobiliários                              |
| `AVG_AMT_FBC_CREDIT_CURRENCY1_BUREAU`| Valor médio dos créditos em moeda local                                  |
| `AVG_AMT_FBC_CREDIT_CURRENCY2_BUREAU`| Valor médio dos créditos em moeda estrangeira                            |
| `MIN_FVL_AMT_CREDIT_SUM_BUREAU`      | Menor valor de crédito registrado                                        |
| `MAX_FVL_AMT_CREDIT_SUM_BUREAU`      | Maior valor de crédito registrado                                        |
| `AVG_FVL_AMT_CREDIT_SUM_BUREAU`      | Valor médio de crédito entre todos os contratos                          |
| `MIN_FVL_DAYS_CREDIT_BUREAU`         | Tempo mínimo desde a concessão de um crédito                             |
| `MAX_FVL_DAYS_CREDIT_BUREAU`         | Tempo máximo desde a concessão de um crédito                             |
| `AVG_FVL_DAYS_CREDIT_BUREAU`         | Tempo médio desde a concessão de crédito                                 |
| `MIN_FVL_AMT_CREDIT_SUM_DEBT_BUREAU` | Menor valor de dívida entre os contratos                                 |
| `MAX_FVL_AMT_CREDIT_SUM_DEBT_BUREAU` | Maior valor de dívida entre os contratos                                 |
| `AVG_FVL_AMT_CREDIT_SUM_DEBT_BUREAU` | Valor médio das dívidas registradas                                      |
| `MIN_FVL_AMT_CREDIT_SUM_OVERDUE_BUREAU`| Menor valor em atraso                                                  |
| `MAX_FVL_AMT_CREDIT_SUM_OVERDUE_BUREAU`| Maior valor em atraso                                                  |
| `AVG_FVL_AMT_CREDIT_SUM_OVERDUE_BUREAU`| Valor médio em atraso                                                  |
| `MIN_FVL_DAYS_CREDIT_UPDATE_BUREAU`  | Menor intervalo desde a última atualização do crédito                    |
| `MAX_FVL_DAYS_CREDIT_UPDATE_BUREAU`  | Maior intervalo desde a última atualização do crédito                    |
| `AVG_FVL_DAYS_CREDIT_UPDATE_BUREAU`  | Tempo médio desde a última atualização do crédito                        |
| `MIN_FVL_CNT_CREDIT_PROLONG_BUREAU`  | Número mínimo de prorrogações                                            |
| `MAX_FVL_CNT_CREDIT_PROLONG_BUREAU`  | Número máximo de prorrogações                                            |
| `AVG_FVL_CNT_CREDIT_PROLONG_BUREAU`  | Número médio de prorrogações                                             |
| `MIN_FVL_AMT_ANNUITY_BUREAU`         | Menor valor de anuidade contratada                                       |
| `MAX_FVL_AMT_ANNUITY_BUREAU`         | Maior valor de anuidade contratada                                       |
| `AVG_FVL_AMT_ANNUITY_BUREAU`         | Valor médio das anuidades                                                |
| `AVG_AMT_CREDIT_ACTIVE_U3M_BUREAU`   | Valor médio de créditos ativos nos últimos 3 meses                       |
| `AVG_AMT_CREDIT_ACTIVE_U6M_BUREAU`   | Valor médio de créditos ativos nos últimos 6 meses                       |
| `AVG_AMT_CREDIT_ACTIVE_U9M_BUREAU`   | Valor médio de créditos ativos nos últimos 9 meses                       |
| `AVG_AMT_CREDIT_ACTIVE_U12M_BUREAU`  | Valor médio de créditos ativos nos últimos 12 meses                      |
| `AVG_RATIO_DEBT_TO_TOTAL_BUREAU`     | Proporção média da dívida sobre o valor total de crédito                 |
| `AVG_RATIO_OVERDUE_TO_TOTAL_BUREAU`  | Proporção média de atraso sobre o valor total de crédito                 |
| `AVG_RATIO_MAXOVERDUE_TO_TOTAL_BUREAU`| Proporção média do maior valor em atraso sobre o crédito total         |
| `AVG_RATIO_DEBT_TO_LIMIT_BUREAU`     | Proporção da dívida em relação ao limite do contrato                     |
| `AVG_RATIO_ANNUITY_TO_DEBT_BUREAU`   | Proporção do valor da anuidade sobre a dívida                            |
| `QT_CONTRACTS_PROLONGED_BUREAU`      | Quantidade de contratos com pelo menos uma prorrogação                   |
| `RATIO_HAS_PROLONG_BUREAU`           | Proporção de contratos prorrogados                                       |
| `AVG_OVERDUE_DAYS_BUREAU`            | Média de dias em atraso                                                  |
| `AVG_DAYS_TO_END_BUREAU`             | Tempo médio até o fim estimado do contrato                               |
| `AVG_DIFF_REAL_VS_ESTIMATED_BUREAU`  | Diferença média entre a data real e estimada de encerramento             |
| `RATIO_EARLY_REPAID_BUREAU`          | Proporção de contratos encerrados antes do previsto                      |


#### Salvando tabela particionada (Parquet)

In [10]:
nm_path = '/data/books/bureau'
bureau_etl_01.write.parquet(nm_path, mode='overwrite')
#bureau_etl_01.coalesce(1).write.parquet(nm_path, mode="overwrite")


In [11]:
spark.stop()